In [4]:
import pandas as pd
import numpy as np

train_ratings = pd.read_csv(
    "../datasets/processed/train_ratings.csv"
)

test_ratings = pd.read_csv(
    "../datasets/processed/test_ratings.csv"
)

recipes = pd.read_csv(
    "../../datasets/RAW_recipes.csv"
)

print("Train shape:", train_ratings.shape)
print("Test shape:", test_ratings.shape)
print("Recipes shape:", recipes.shape)

Train shape: (428067, 5)
Test shape: (106980, 5)
Recipes shape: (231637, 12)


In [2]:
import os

print(os.getcwd())

C:\Users\Nikola\Desktop\FoodRecommendationSystem\ai\notebooks


In [5]:
recipe_popularity = (
    train_ratings
    .groupby("recipe_id")
    .size()
    .reset_index(name="rating_count")
)

recipe_popularity = recipe_popularity.sort_values(
    "rating_count",
    ascending=False
)

recipe_popularity.head(10)

,recipe_id,rating_count
3997,27208,802
13799,89204,782
6083,39087,663
4947,32204,649
10992,69173,580
3178,22782,543
8558,54257,529
3732,25885,504
12709,82102,504
10959,68955,498


In [6]:
top_popular_recipes = recipe_popularity.merge(
    recipes[["id", "name"]],
    left_on="recipe_id",
    right_on="id",
    how="left"
)

top_popular_recipes = top_popular_recipes[
    ["recipe_id", "name", "rating_count"]
].head(10)

top_popular_recipes

,recipe_id,name,rating_count
0,27208,to die for crock pot roast,802
1,89204,crock pot chicken with black beans cream cheese,782
2,39087,creamy cajun chicken pasta,663
3,32204,whatever floats your boat brownies,649
4,69173,kittencal s italian melt in your mouth meatballs,580
5,22782,jo mama s world famous spaghetti,543
6,54257,yes virginia there is a great meatloaf,529
7,25885,banana banana bread,504
8,82102,kittencal s moist cheddar garlic oven fried ch...,504
9,68955,japanese mum s chicken,498


In [7]:
def recommend_popular(user_id, n=10):
    """
    Recommend the N most popular recipes
    that the user has not already rated.
    """

    user_rated = set(
        train_ratings.loc[
            train_ratings["user_id"] == user_id,
            "recipe_id"
        ]
    )

    recommendations = recipe_popularity[
        ~recipe_popularity["recipe_id"].isin(user_rated)
    ].head(n)

    return recommendations

In [8]:
test_user_id = test_ratings["user_id"].iloc[0]

print("Test user:", test_user_id)

recommendations = recommend_popular(
    test_user_id,
    n=10
)

recommendations

Test user: 162826


,recipe_id,rating_count
3997,27208,802
13799,89204,782
6083,39087,663
4947,32204,649
3178,22782,543
8558,54257,529
3732,25885,504
12709,82102,504
10959,68955,498
4164,28148,491


In [9]:
def precision_at_k(user_id, recommendations, k=10):
    """
    Calculate Precision@K for one user.
    """

    actual = set(
        test_ratings.loc[
            test_ratings["user_id"] == user_id,
            "recipe_id"
        ]
    )

    recommended = set(
        recommendations.head(k)["recipe_id"]
    )

    hits = len(actual.intersection(recommended))

    return hits / k

In [10]:
precision = precision_at_k(
    test_user_id,
    recommendations,
    k=10
)

print("Precision@10:", precision)

Precision@10: 0.2


In [11]:
def evaluate_popularity_model(k=10):
    precisions = []

    test_users = test_ratings["user_id"].unique()

    for user_id in test_users:
        recommendations = recommend_popular(user_id, n=k)

        precision = precision_at_k(
            user_id,
            recommendations,
            k=k
        )

        precisions.append(precision)

    return np.mean(precisions)

In [12]:
precision_at_10 = evaluate_popularity_model(k=10)

print("Popularity Baseline Precision@10:", precision_at_10)

Popularity Baseline Precision@10: 0.010804799137233756


In [13]:
def recall_at_k(user_id, recommendations, k=10):
    """
    Calculate Recall@K for one user.
    """

    actual = set(
        test_ratings.loc[
            test_ratings["user_id"] == user_id,
            "recipe_id"
        ]
    )

    recommended = set(
        recommendations.head(k)["recipe_id"]
    )

    if len(actual) == 0:
        return 0.0

    hits = len(actual.intersection(recommended))

    return hits / len(actual)

In [14]:
def evaluate_recall_popularity_model(k=10):
    recalls = []

    test_users = test_ratings["user_id"].unique()

    for user_id in test_users:
        recommendations = recommend_popular(user_id, n=k)

        recall = recall_at_k(
            user_id,
            recommendations,
            k=k
        )

        recalls.append(recall)

    return np.mean(recalls)

In [15]:
recall_at_10 = evaluate_recall_popularity_model(k=10)

print("Popularity Baseline Recall@10:", recall_at_10)

Popularity Baseline Recall@10: 0.02509049346872046


In [17]:
import pickle

popularity_model = {
    "recipe_popularity": recipe_popularity,
    "k": 10,
    "precision_at_10": precision_at_10,
    "recall_at_10": recall_at_10
}
with open("../saved_models/popularity_baseline.pkl", "wb") as f:
    pickle.dump(popularity_model, f)

print("Popularity Baseline model saved.")

Popularity Baseline model saved.


In [18]:


with open("../saved_models/popularity_baseline.pkl", "rb") as f:
    loaded_model = pickle.load(f)

print("Model successfully loaded.")
print("Precision@10:", loaded_model["precision_at_10"])
print("Recall@10:", loaded_model["recall_at_10"])

Model successfully loaded.
Precision@10: 0.010804799137233756
Recall@10: 0.02509049346872046


In [19]:
def recommend_from_saved_model(user_id, n=10):
    """
    Generate recommendations using the saved Popularity Baseline model.
    """

    popularity = loaded_model["recipe_popularity"]

    user_rated = set(
        train_ratings.loc[
            train_ratings["user_id"] == user_id,
            "recipe_id"
        ]
    )

    recommendations = popularity[
        ~popularity["recipe_id"].isin(user_rated)
    ].head(n)

    return recommendations

In [20]:
saved_recommendations = recommend_from_saved_model(
    user_id=162826,
    n=10
)

saved_recommendations

,recipe_id,rating_count
3997,27208,802
13799,89204,782
6083,39087,663
4947,32204,649
3178,22782,543
8558,54257,529
3732,25885,504
12709,82102,504
10959,68955,498
4164,28148,491
